# 1. Objective

Notebook ini membangun **model TensorFlow/Keras klasifikasi 3 kelas** untuk data finansial tabular.

## Target (resmi dari dataset)
- `TARGET = "financial_scenario"`
- Mapping kelas (English output):

```python
{0: "inflation", 1: "normal", 2: "recession"}
```

## Final artifacts
- `artifacts/classification_model.keras`
- `artifacts/classification_scaler.joblib`
- `artifacts/classification_feature_columns.json`
- `artifacts/classification_label_mapping.json`
- `artifacts/classification_metrics.json`


## 2. Import Libraries

Library yang dipakai:
- Data: pandas, numpy
- Preprocessing: scikit-learn (split, RobustScaler)
- Deep Learning: TensorFlow/Keras
- Evaluation: confusion matrix, classification report, macro/weighted F1

Catatan: TensorBoard logging **dinonaktifkan** (tidak membuat folder log).

In [258]:
import json
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score

warnings.filterwarnings("ignore")
print("TensorFlow:", tf.__version__)


TensorFlow: 2.21.0


## 3. Configuration

Konfigurasi dipakai konsisten untuk training, evaluasi, dan artifact saving (reproducible).


In [259]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = Path("FINARY_FIXED_FINAL_DATASET.csv")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "financial_scenario"
LABEL_MAPPING = {"0": "inflation", "1": "normal", "2": "recession"}

# Production unit scaling (consistency with API integration)
UNIT_SCALE = 2500.0

# Training controls
EPOCHS = 120
BATCH_SIZE = 128
PATIENCE = 15
LEARNING_RATE = 1e-3

# Sedikit dinaikkan untuk mengurangi overfitting (gap train>>test)
WEIGHT_DECAY = 1e-4

CLIP_NORM = 1.0

# Optional (keep 0.0 by default)
LABEL_SMOOTHING = 0.1

print("DATA_PATH:", DATA_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("TARGET:", TARGET)
print("LABEL_MAPPING:", LABEL_MAPPING)
print("WEIGHT_DECAY:", WEIGHT_DECAY)


DATA_PATH: FINARY_FIXED_FINAL_DATASET.csv
ARTIFACT_DIR: artifacts
TARGET: financial_scenario
LABEL_MAPPING: {'0': 'inflation', '1': 'normal', '2': 'recession'}
WEIGHT_DECAY: 0.0001


## 4. Load Dataset

Load dataset utama `FINARY_FIXED_FINAL_DATASET.csv`.


In [260]:
df = pd.read_csv(DATA_PATH)
print("Rows:", len(df), "Cols:", df.shape[1])
df.head(3)


Rows: 3000 Cols: 43


,monthly_income,monthly_expense_total,savings_rate,budget_goal,financial_scenario,credit_score,debt_to_income_ratio,loan_payment,investment_amount,subscription_services,...,category_Healthcare,category_Insurance,category_Investments,category_Rent,category_Transportation,category_Utilities,cash_flow_status_Neutral,cash_flow_status_Positive,financial_stress_level_Low,financial_stress_level_Medium
0,3119.58,3212.07,0.38,3676.11,0,721.0,0.56,125.77,689.22,3,...,False,False,True,False,False,False,False,True,True,False
1,3262.44,3732.81,0.10,2607.17,0,670.0,0.42,454.19,360.34,4,...,False,False,True,False,False,False,False,True,True,False
2,2931.20,3335.58,0.15,3004.14,0,691.0,0.24,971.82,0.00,5,...,True,False,False,False,False,False,False,True,True,False


## 5. Data Audit

Tujuan audit:
- cek tipe data & range nilai
- cek kolom boolean/one-hot
- cek missing value


In [261]:
display(df.dtypes.value_counts())
display(df.describe(include="all").T.head(15))

missing = df.isna().mean().sort_values(ascending=False)
display(missing.head(15))


float64    21
bool       15
int64       7
Name: count, dtype: int64

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
monthly_income,3000.0,NaN,NaN,NaN,4004.267347,1000.107096,685.28,3362.25,4008.075,4659.04,7407.94
monthly_expense_total,3000.0,NaN,NaN,NaN,3011.68223,801.151864,159.21,2473.205,3023.81,3555.5025,5853.2
savings_rate,3000.0,NaN,NaN,NaN,0.225897,0.101417,0.05,0.14,0.23,0.31,0.4
budget_goal,3000.0,NaN,NaN,NaN,2811.070463,490.855344,1175.57,2481.9575,2822.285,3131.0,4386.5
financial_scenario,3000.0,NaN,NaN,NaN,0.988333,0.648334,0.0,1.0,1.0,1.0,2.0
credit_score,3000.0,NaN,NaN,NaN,679.923667,49.970847,515.0,646.0,679.0,713.0,847.0
debt_to_income_ratio,3000.0,NaN,NaN,NaN,0.350817,0.145191,0.1,0.22,0.35,0.48,0.6
loan_payment,3000.0,NaN,NaN,NaN,508.58113,199.44458,0.0,375.34,508.57,638.47,1176.88
investment_amount,3000.0,NaN,NaN,NaN,400.610493,235.361273,0.0,234.78,388.075,559.8425,1292.3
subscription_services,3000.0,NaN,NaN,NaN,4.975,2.559536,1.0,3.0,5.0,7.0,9.0


monthly_income            0.0
monthly_expense_total     0.0
savings_rate              0.0
budget_goal               0.0
financial_scenario        0.0
credit_score              0.0
debt_to_income_ratio      0.0
loan_payment              0.0
investment_amount         0.0
subscription_services     0.0
emergency_fund            0.0
transaction_count         0.0
fraud_flag                0.0
discretionary_spending    0.0
essential_spending        0.0
dtype: float64

## 6. Target Distribution & Imbalance Check

Validasi label target hanya `{0,1,2}`, lalu cek distribusi kelas (imbalance).


In [262]:
unique_targets = sorted(df[TARGET].dropna().unique().tolist())
print("Unique targets:", unique_targets)
if unique_targets != [0, 1, 2]:
    raise ValueError(f"Invalid target values for {TARGET}. Expected [0,1,2], got {unique_targets}")

counts = df[TARGET].value_counts().sort_index()
print("Class distribution (full dataset):")
print(counts)

majority_class_full = int(counts.idxmax())
majority_ratio_full = float(counts.max() / counts.sum())
print("Majority class (full):", majority_class_full, "(", LABEL_MAPPING[str(majority_class_full)], ")")
print("Majority ratio (full):", round(majority_ratio_full, 4))


Unique targets: [0, 1, 2]
Class distribution (full dataset):
financial_scenario
0     648
1    1739
2     613
Name: count, dtype: int64
Majority class (full): 1 ( normal )
Majority ratio (full): 0.5797


## 7. Data Cleaning

Aturan cleaning:
- boolean → integer (0/1)
- missing values → median (numerik)
- drop kolom konstan
- drop kolom duplikat sempurna
- pastikan target tidak masuk fitur
- pastikan seluruh fitur numeric


In [263]:
df_clean = df.copy()

# bool -> int
for col in df_clean.columns:
    if df_clean[col].dtype == bool:
        df_clean[col] = df_clean[col].astype(int)

# ensure target numeric int
if not np.issubdtype(df_clean[TARGET].dtype, np.number):
    raise ValueError(f"Target dtype is not numeric: {df_clean[TARGET].dtype}")
df_clean[TARGET] = df_clean[TARGET].astype(int)

# numeric feature candidates (exclude target)
num_cols = df_clean.drop(columns=[TARGET]).select_dtypes(include=["number"]).columns.tolist()

# missing -> median
missing_before = int(df_clean[num_cols].isna().sum().sum())
medians = df_clean[num_cols].median(numeric_only=True)
df_clean[num_cols] = df_clean[num_cols].fillna(medians)
missing_after = int(df_clean[num_cols].isna().sum().sum())
print("Missing numeric values before:", missing_before)
print("Missing numeric values after :", missing_after)

# drop constant
unique_counts = df_clean[num_cols].nunique(dropna=False)
constant_cols = unique_counts[unique_counts <= 1].index.tolist()
print("Constant columns:", len(constant_cols))

# drop perfect duplicates
dup_mask = df_clean[num_cols].T.duplicated()
duplicate_cols = dup_mask[dup_mask].index.tolist()
print("Duplicate columns:", len(duplicate_cols))

excluded_cols = sorted(set(constant_cols + duplicate_cols))
feature_cols_clean = [c for c in num_cols if c not in excluded_cols]

X_full = df_clean[feature_cols_clean].astype("float32")
y_full = df_clean[TARGET].astype(int)

assert TARGET not in X_full.columns
assert len(X_full) == len(y_full)

print("Feature columns after cleaning:", len(feature_cols_clean))
X_full.head(3)


Missing numeric values before: 0
Missing numeric values after : 0
Constant columns: 0
Duplicate columns: 0
Feature columns after cleaning: 42


,monthly_income,monthly_expense_total,savings_rate,budget_goal,credit_score,debt_to_income_ratio,loan_payment,investment_amount,subscription_services,emergency_fund,...,category_Healthcare,category_Insurance,category_Investments,category_Rent,category_Transportation,category_Utilities,cash_flow_status_Neutral,cash_flow_status_Positive,financial_stress_level_Low,financial_stress_level_Medium
0,3119.580078,3212.070068,0.38,3676.110107,721.0,0.56,125.769997,689.219971,3.0,510.579987,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,3262.439941,3732.810059,0.10,2607.169922,670.0,0.42,454.190002,360.339996,4.0,1154.410034,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2,2931.199951,3335.580078,0.15,3004.139893,691.0,0.24,971.820007,0.000000,5.0,1433.020020,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


## 8. Feature Engineering (Dinonaktifkan)

Sesuai optimasi terbaru, kita **tidak menambah engineered features**. Kita pakai semua fitur valid dari dataset (setelah cleaning) agar distribusi fitur tetap natural dan konsisten dengan data asli.


In [264]:
# Engineered features dinonaktifkan.
# Best practice untuk SMOTE pada data tabular yang punya banyak one-hot:
# tetap gunakan fitur asli (setelah cleaning) untuk menjaga distribusi biner one-hot.
X_eng = X_full.copy().select_dtypes(include=["number"]).astype("float32")

print("Feature count (cleaned):", X_eng.shape[1])
X_eng.head(3)


Feature count (cleaned): 42


,monthly_income,monthly_expense_total,savings_rate,budget_goal,credit_score,debt_to_income_ratio,loan_payment,investment_amount,subscription_services,emergency_fund,...,category_Healthcare,category_Insurance,category_Investments,category_Rent,category_Transportation,category_Utilities,cash_flow_status_Neutral,cash_flow_status_Positive,financial_stress_level_Low,financial_stress_level_Medium
0,3119.580078,3212.070068,0.38,3676.110107,721.0,0.56,125.769997,689.219971,3.0,510.579987,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,3262.439941,3732.810059,0.10,2607.169922,670.0,0.42,454.190002,360.339996,4.0,1154.410034,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2,2931.199951,3335.580078,0.15,3004.139893,691.0,0.24,971.820007,0.000000,5.0,1433.020020,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


## 9. Leakage & Feature Validity Check

Pastikan:
- target tidak masuk fitur
- seluruh fitur numeric


In [265]:
# Gunakan semua feature numeric valid (setelah cleaning)
feature_columns = X_eng.columns.tolist()

assert TARGET not in feature_columns

non_numeric = [c for c in X_eng.columns if not np.issubdtype(X_eng[c].dtype, np.number)]
if non_numeric:
    raise ValueError(f"Non-numeric features found: {non_numeric}")

print("Final feature_count:", len(feature_columns))
print("Sample feature columns:", feature_columns[:20])


Final feature_count: 42
Sample feature columns: ['monthly_income', 'monthly_expense_total', 'savings_rate', 'budget_goal', 'credit_score', 'debt_to_income_ratio', 'loan_payment', 'investment_amount', 'subscription_services', 'emergency_fund', 'transaction_count', 'fraud_flag', 'discretionary_spending', 'essential_spending', 'rent_or_mortgage', 'financial_advice_score', 'actual_savings', 'savings_goal_met', 'net_cash_flow', 'expense_ratio']


## 10. Train/Validation/Test Split

Stratified split 70/15/15 untuk menjaga distribusi label.


In [266]:
X_all = X_eng
assert len(X_all) == len(y_full)

X_train_df, X_temp_df, y_train, y_temp = train_test_split(
    X_all,
    y_full,
    test_size=0.30,
    random_state=SEED,
    stratify=y_full,
)

X_val_df, X_test_df, y_val, y_test = train_test_split(
    X_temp_df,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

def label_dist(arr: pd.Series) -> dict:
    vc = arr.value_counts().sort_index()
    return {int(k): int(v) for k, v in vc.items()}

print("Shapes:")
print("  Train:", X_train_df.shape, y_train.shape)
print("  Val  :", X_val_df.shape, y_val.shape)
print("  Test :", X_test_df.shape, y_test.shape)
print("Label distribution:")
print("  Train:", label_dist(y_train))
print("  Val  :", label_dist(y_val))
print("  Test :", label_dist(y_test))

majority_class = int(y_train.value_counts().idxmax())
majority_baseline_acc = float((y_test.values == majority_class).mean())
print("Majority baseline accuracy (test):", round(majority_baseline_acc, 4))


Shapes:
  Train: (2100, 42) (2100,)
  Val  : (450, 42) (450,)
  Test : (450, 42) (450,)
Label distribution:
  Train: {0: 454, 1: 1217, 2: 429}
  Val  : {0: 97, 1: 261, 2: 92}
  Test : {0: 97, 1: 261, 2: 92}
Majority baseline accuracy (test): 0.58


## 11. Imbalance Handling

Gunakan:
- class weights (balanced)
- optional oversampling train-only (jika recall minor class buruk)


In [267]:
# Class weights (lebih konservatif) untuk mengurangi normal tertarik ke ekstrem.
class_weight = {0: 1.5, 1: 1.0, 2: 1.5}
print("Class weights:", class_weight)


Class weights: {0: 1.5, 1: 1.0, 2: 1.5}


## 12. Preprocessing

Gunakan `RobustScaler`, fit hanya pada train.


In [268]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_df.values)
X_val_scaled = scaler.transform(X_val_df.values)
X_test_scaled = scaler.transform(X_test_df.values)

# Tidak ada SMOTE/Tomek: train/val/test tetap pada distribusi asli.
input_dim = X_train_scaled.shape[1]
print("input_dim:", input_dim)


input_dim: 42


## 13. Final Model Architecture

Residual MLP / ResNet-style Tabular Network (Functional API + custom residual block).


In [269]:
class ResidualDenseBlock(tf.keras.layers.Layer):
    """Residual block yang ringan untuk tabular (custom component)."""

    def __init__(self, units: int, dropout: float, l2: float, activation: str = "gelu", name: str | None = None):
        super().__init__(name=name)
        reg = tf.keras.regularizers.l2(l2)
        self.units = units
        self.activation = activation

        self.d1 = tf.keras.layers.Dense(units, kernel_regularizer=reg)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.dp1 = tf.keras.layers.Dropout(dropout)

        self.d2 = tf.keras.layers.Dense(units, kernel_regularizer=reg)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.dp2 = tf.keras.layers.Dropout(dropout)

        self.proj = None

    def _act(self, x):
        return tf.keras.activations.gelu(x) if self.activation.lower() == "gelu" else tf.nn.relu(x)

    def build(self, input_shape):
        in_units = int(input_shape[-1])
        if in_units != self.units:
            self.proj = tf.keras.layers.Dense(self.units)
        super().build(input_shape)

    def call(self, x, training=False):
        skip = self.proj(x) if self.proj is not None else x

        y = self.d1(x)
        y = self.bn1(y, training=training)
        y = self._act(y)
        y = self.dp1(y, training=training)

        y = self.d2(y)
        y = self.bn2(y, training=training)
        y = self._act(y)
        y = self.dp2(y, training=training)

        return skip + y


def build_residual_mlp(input_dim: int) -> tf.keras.Model:
    """Model disederhanakan: cukup 2 residual blocks + head kecil."""

    reg = tf.keras.regularizers.l2(WEIGHT_DECAY)
    inputs = tf.keras.Input(shape=(input_dim,), name="classification_input")

    x = tf.keras.layers.Dense(128, kernel_regularizer=reg, name="proj_dense")(inputs)
    x = tf.keras.layers.BatchNormalization(name="proj_bn")(x)
    x = tf.keras.layers.Activation(tf.keras.activations.gelu, name="proj_act")(x)
    x = tf.keras.layers.Dropout(0.20, name="proj_dropout")(x)

    x = ResidualDenseBlock(128, dropout=0.20, l2=WEIGHT_DECAY, activation="gelu", name="res_block_1")(x)
    x = ResidualDenseBlock(64, dropout=0.15, l2=WEIGHT_DECAY, activation="gelu", name="res_block_2")(x)

    x = tf.keras.layers.BatchNormalization(name="head_bn")(x)
    x = tf.keras.layers.Dense(64, kernel_regularizer=reg, name="head_dense_64")(x)
    x = tf.keras.layers.Activation(tf.keras.activations.gelu, name="head_act")(x)
    x = tf.keras.layers.Dropout(0.15, name="head_dropout")(x)

    outputs = tf.keras.layers.Dense(3, activation="softmax", name="classification_output")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="finary_classification_residual_mlp")


model = build_residual_mlp(input_dim)
model.summary()


Model: "finary_classification_residual_mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ classification_input            │ (None, 42)             │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj_dense (Dense)              │ (None, 128)            │         5,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj_bn (BatchNormalization)    │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj_act (Activation)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj_dropout (Dropout)          │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ res_block_1                     │ (None, 128)            │        34,048 │
│ (ResidualDenseBlock)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ res_block_2                     │ (None, 64)             │        21,184 │
│ (ResidualDenseBlock)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_bn (BatchNormalization)    │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dense_64 (Dense)           │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_act (Activation)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout (Dropout)          │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification_output (Dense)   │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,859 (257.26 KB)

 Trainable params: 64,707 (252.76 KB)

 Non-trainable params: 1,152 (4.50 KB)

## 14. Custom Loss / Custom Component

Gunakan **Weighted Focal Loss** (multi-class) untuk membantu imbalance + contoh sulit.


In [270]:
def to_onehot(y: np.ndarray, num_classes: int = 3) -> np.ndarray:
    y = y.astype(int)
    out = np.zeros((len(y), num_classes), dtype=np.float32)
    out[np.arange(len(y)), y] = 1.0
    return out


# Weighted Cross-Entropy (lebih stabil daripada focal untuk kasus overlap antar kelas)
_ce = tf.keras.losses.CategoricalCrossentropy(from_logits=False, reduction=tf.keras.losses.Reduction.NONE)


def weighted_categorical_crossentropy(class_weight_dict: dict[int, float]):
    w = np.array([class_weight_dict[i] for i in range(3)], dtype=np.float32)
    w = w * (3.0 / float(w.sum()))  # normalize for stability
    w_t = tf.constant(w, dtype=tf.float32)

    def loss(y_true, y_pred):
        # per-sample CE
        ce = _ce(y_true, y_pred)
        # pick weight by true class (one-hot)
        sample_w = tf.reduce_sum(y_true * w_t, axis=-1)
        return sample_w * ce

    return loss


loss_fn = weighted_categorical_crossentropy(class_weight)
print("Custom loss ready: weighted categorical cross-entropy")
print("Class weight (normalized):", (np.array([class_weight[0], class_weight[1], class_weight[2]], dtype=np.float32) * (3.0 / float(sum(class_weight.values())))).round(4))


Custom loss ready: weighted categorical cross-entropy
Class weight (normalized): [1.125 0.75  1.125]


## 15. Custom Training Loop with tf.GradientTape

Training loop manual mencakup:
- `train_step` & `val_step`
- manual epoch loop
- early stopping + best weights restore
- metric tracking (loss, accuracy, val_macro_f1)


In [271]:
def make_tf_dataset(X: np.ndarray, y_onehot: np.ndarray, batch_size: int, training: bool) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((X.astype(np.float32), y_onehot.astype(np.float32)))
    if training:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


def macro_f1_from_probs(y_true: np.ndarray, probs: np.ndarray) -> float:
    y_pred = np.argmax(probs, axis=1)
    return float(f1_score(y_true, y_pred, average="macro"))


y_train_oh = to_onehot(y_train.values, 3)
y_val_oh = to_onehot(y_val.values, 3)
y_test_oh = to_onehot(y_test.values, 3)

train_ds = make_tf_dataset(X_train_scaled, y_train_oh, BATCH_SIZE, training=True)
val_ds = make_tf_dataset(X_val_scaled, y_val_oh, BATCH_SIZE, training=False)
test_ds = make_tf_dataset(X_test_scaled, y_test_oh, BATCH_SIZE, training=False)


optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=CLIP_NORM)
train_acc_m = tf.keras.metrics.CategoricalAccuracy(name="train_accuracy")
val_acc_m = tf.keras.metrics.CategoricalAccuracy(name="val_accuracy")


@tf.function
def train_step(xb, yb):
    with tf.GradientTape() as tape:
        probs = model(xb, training=True)
        if LABEL_SMOOTHING and LABEL_SMOOTHING > 0:
            yb_ls = yb * (1.0 - LABEL_SMOOTHING) + (LABEL_SMOOTHING / 3.0)
        else:
            yb_ls = yb
        loss_vals = loss_fn(yb_ls, probs)
        loss = tf.reduce_mean(loss_vals)
        if model.losses:
            loss += tf.add_n(model.losses)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    train_acc_m.update_state(yb, probs)
    return loss


@tf.function
def val_step(xb, yb):
    probs = model(xb, training=False)
    loss_vals = loss_fn(yb, probs)
    loss = tf.reduce_mean(loss_vals)
    if model.losses:
        loss += tf.add_n(model.losses)
    val_acc_m.update_state(yb, probs)
    return loss, probs


## 16. Training

TensorBoard logging **dinonaktifkan** (sesuai permintaan) agar notebook tidak membuat folder log.

Training tetap memakai custom loop `tf.GradientTape` dengan early stopping berbasis **`val_macro_f1`** dan restore best weights.


In [272]:
# TensorBoard logging dinonaktifkan
TB_LOG_DIR = None
print("TensorBoard logging: disabled")


TensorBoard logging: disabled


(Bagian training ada di cell berikutnya.)


In [273]:
best_val_macro_f1 = -1.0
best_epoch = 0
best_weights = None
bad_epochs = 0

history = []

for epoch in range(1, EPOCHS + 1):
    train_acc_m.reset_state()
    val_acc_m.reset_state()

    # Train
    train_losses = []
    for xb, yb in train_ds:
        loss = train_step(xb, yb)
        train_losses.append(float(loss.numpy()))

    # Val
    val_losses = []
    val_probs_all = []
    val_y_all = []
    for xb, yb in val_ds:
        vloss, probs = val_step(xb, yb)
        val_losses.append(float(vloss.numpy()))
        val_probs_all.append(probs.numpy())
        val_y_all.append(np.argmax(yb.numpy(), axis=1))

    val_probs_all = np.concatenate(val_probs_all, axis=0)
    val_y_all = np.concatenate(val_y_all, axis=0)

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    train_acc = float(train_acc_m.result().numpy())
    val_acc = float(val_acc_m.result().numpy())
    val_macro_f1 = macro_f1_from_probs(val_y_all, val_probs_all)

    lr = float(optimizer.learning_rate.numpy())

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "val_macro_f1": val_macro_f1,
            "learning_rate": lr,
        }
    )

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_macro_f1={val_macro_f1:.4f} | "
        f"lr={lr:.6f}"
    )

    # Early stopping on val_macro_f1 + restore best weights
    if val_macro_f1 > best_val_macro_f1 + 1e-4:
        best_val_macro_f1 = val_macro_f1
        best_epoch = epoch
        best_weights = model.get_weights()
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (best_epoch={best_epoch}, best_val_macro_f1={best_val_macro_f1:.4f})")
        break

if best_weights is not None:
    model.set_weights(best_weights)


Epoch 001 | train_loss=1.1937 train_acc=0.3781 | val_loss=1.0361 val_acc=0.5222 val_macro_f1=0.2864 | lr=0.001000
Epoch 002 | train_loss=1.1059 train_acc=0.4714 | val_loss=1.0339 val_acc=0.5400 val_macro_f1=0.2455 | lr=0.001000
Epoch 003 | train_loss=1.0654 train_acc=0.5090 | val_loss=1.0321 val_acc=0.5711 val_macro_f1=0.2423 | lr=0.001000
Epoch 004 | train_loss=1.0482 train_acc=0.5281 | val_loss=1.0288 val_acc=0.5711 val_macro_f1=0.2427 | lr=0.001000
Epoch 005 | train_loss=1.0362 train_acc=0.5129 | val_loss=1.0281 val_acc=0.5689 val_macro_f1=0.2421 | lr=0.001000
Epoch 006 | train_loss=1.0308 train_acc=0.5076 | val_loss=1.0267 val_acc=0.5711 val_macro_f1=0.2430 | lr=0.001000
Epoch 007 | train_loss=1.0071 train_acc=0.5371 | val_loss=1.0272 val_acc=0.5733 val_macro_f1=0.2436 | lr=0.001000
Epoch 008 | train_loss=0.9959 train_acc=0.5610 | val_loss=1.0275 val_acc=0.5667 val_macro_f1=0.2484 | lr=0.001000
Epoch 009 | train_loss=1.0041 train_acc=0.5533 | val_loss=1.0285 val_acc=0.5578 val_macr

## 17. Evaluation

Evaluasi mencakup:
- train/val/test accuracy
- macro F1 & weighted F1
- per-class precision/recall/F1
- confusion matrix + classification report
- majority baseline comparison


In [274]:
def predict_probs(ds: tf.data.Dataset) -> tuple[np.ndarray, np.ndarray]:
    probs_all = []
    y_all = []
    for xb, yb in ds:
        probs = model(xb, training=False).numpy()
        probs_all.append(probs)
        y_all.append(np.argmax(yb.numpy(), axis=1))
    return np.concatenate(y_all, axis=0), np.concatenate(probs_all, axis=0)


y_tr, p_tr = predict_probs(train_ds)
y_va, p_va = predict_probs(val_ds)
y_te, p_te = predict_probs(test_ds)

def acc(y_true, probs):
    return float((np.argmax(probs, axis=1) == y_true).mean())

train_accuracy = acc(y_tr, p_tr)
val_accuracy = acc(y_va, p_va)
test_accuracy = acc(y_te, p_te)

macro_f1 = float(f1_score(y_te, np.argmax(p_te, axis=1), average="macro"))
weighted_f1 = float(f1_score(y_te, np.argmax(p_te, axis=1), average="weighted"))

cm = confusion_matrix(y_te, np.argmax(p_te, axis=1), labels=[0, 1, 2]).tolist()
report = classification_report(y_te, np.argmax(p_te, axis=1), labels=[0, 1, 2], output_dict=True, zero_division=0)

print("Train accuracy:", round(train_accuracy, 4))
print("Val accuracy  :", round(val_accuracy, 4))
print("Test accuracy :", round(test_accuracy, 4))
print("Macro F1      :", round(macro_f1, 4))
print("Weighted F1   :", round(weighted_f1, 4))
print("Confusion matrix:", cm)


Train accuracy: 0.5357
Val accuracy  : 0.5222
Test accuracy : 0.5333
Macro F1      : 0.277
Weighted F1   : 0.4356
Confusion matrix: [[2, 79, 16], [4, 232, 25], [1, 85, 6]]


## 18. Error Analysis

Analisis:
- kelas mana paling sering tertukar
- bias ke normal
- contoh 10 prediksi salah
- confidence (max prob) pada prediksi salah


In [275]:
y_pred = np.argmax(p_te, axis=1)
conf = np.max(p_te, axis=1)

errors = np.where(y_pred != y_te)[0]
print("Total test errors:", len(errors), "out of", len(y_te))

if len(errors) > 0:
    # Top confusion pairs
    pair_counts = {}
    for idx in errors:
        pair = (int(y_te[idx]), int(y_pred[idx]))
        pair_counts[pair] = pair_counts.get(pair, 0) + 1
    pair_sorted = sorted(pair_counts.items(), key=lambda kv: kv[1], reverse=True)
    print("Most common confusions (true->pred):")
    for (t, p), n in pair_sorted[:10]:
        print(f"  {t}->{p} ({LABEL_MAPPING[str(t)]}->{LABEL_MAPPING[str(p)]}): {n}")

    # Show 10 wrong examples with confidence
    show_n = min(10, len(errors))
    sample_err = errors[:show_n]

    err_rows = X_test_df.iloc[sample_err].copy()
    err_rows["true"] = [LABEL_MAPPING[str(int(v))] for v in y_te[sample_err]]
    err_rows["pred"] = [LABEL_MAPPING[str(int(v))] for v in y_pred[sample_err]]
    err_rows["confidence"] = conf[sample_err]
    display(err_rows[["true", "pred", "confidence"]].head(show_n))


Total test errors: 210 out of 450
Most common confusions (true->pred):
  2->1 (recession->normal): 85
  0->1 (inflation->normal): 79
  1->2 (normal->recession): 25
  0->2 (inflation->recession): 16
  1->0 (normal->inflation): 4
  2->0 (recession->inflation): 1


,true,pred,confidence
2820,inflation,normal,0.424562
1841,recession,normal,0.375594
2258,recession,normal,0.342432
1123,inflation,normal,0.407459
829,recession,normal,0.436761
1446,normal,recession,0.361921
199,inflation,normal,0.371382
2238,recession,normal,0.369360
2268,normal,recession,0.358863
1089,normal,inflation,0.335218


## 19. Save Artifacts

Simpan hanya artifacts final:
- model `.keras`
- scaler `.joblib`
- feature columns `.json`
- label mapping `.json`
- metrics `.json`


In [276]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

model_path = ARTIFACT_DIR / "classification_model.keras"
scaler_path = ARTIFACT_DIR / "classification_scaler.joblib"
feat_cols_path = ARTIFACT_DIR / "classification_feature_columns.json"
label_map_path = ARTIFACT_DIR / "classification_label_mapping.json"
metrics_path = ARTIFACT_DIR / "classification_metrics.json"

model.save(model_path)
joblib.dump(scaler, scaler_path)

with open(feat_cols_path, "w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)

with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump(LABEL_MAPPING, f, indent=2)

per_class_precision = {str(k): float(report[str(k)]["precision"]) for k in [0, 1, 2]}
per_class_recall = {str(k): float(report[str(k)]["recall"]) for k in [0, 1, 2]}
per_class_f1 = {str(k): float(report[str(k)]["f1-score"]) for k in [0, 1, 2]}

metrics = {
    "model_name": model.name,
    "target_mapping": LABEL_MAPPING,
    "feature_count": int(len(feature_columns)),
    "feature_columns": feature_columns,
    "class_distribution": {
        "train": label_dist(y_train),
        "val": label_dist(y_val),
        "test": label_dist(y_test),
    },
    "majority_baseline_accuracy": float(majority_baseline_acc),
    "train_accuracy": float(train_accuracy),
    "val_accuracy": float(val_accuracy),
    "test_accuracy": float(test_accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "per_class_precision": per_class_precision,
    "per_class_recall": per_class_recall,
    "per_class_f1": per_class_f1,
    "confusion_matrix": cm,
    "loss_function": "weighted_categorical_crossentropy(class_weight)",
    "optimizer": "Adam(clipnorm=CLIP_NORM)",
    "epochs_trained": int(len(history)),
    "best_epoch": int(best_epoch),
    "tensorboard_log_dir": None,
    "notes": "TensorBoard logging dinonaktifkan. Early stopping berbasis val_macro_f1 + restore best weights.",
}

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Saved:")
print(" -", model_path)
print(" -", scaler_path)
print(" -", feat_cols_path)
print(" -", label_map_path)
print(" -", metrics_path)


Saved:
 - artifacts\classification_model.keras
 - artifacts\classification_scaler.joblib
 - artifacts\classification_feature_columns.json
 - artifacts\classification_label_mapping.json
 - artifacts\classification_metrics.json


## 20. Local Inference Test

Uji inference lokal (mirip flow FastAPI):
- input monetary dalam IDR
- scale monetary fields dengan `UNIT_SCALE`
- reindex sesuai `classification_feature_columns.json`
- scaler transform
- model predict → output class + probabilities


In [277]:
def build_features_from_raw_idr(payload: dict) -> pd.DataFrame:
    # Convert IDR -> unit scale used by dataset
    monetary_fields = [
        "monthly_income",
        "monthly_expense_total",
        "budget_goal",
        "loan_payment",
        "investment_amount",
        "emergency_fund",
        "discretionary_spending",
        "essential_spending",
        "rent_or_mortgage",
        "actual_savings",
    ]
    row = payload.copy()
    for k in monetary_fields:
        if k in row:
            row[k] = float(row[k]) / UNIT_SCALE

    df_row = pd.DataFrame([row]).astype("float32", errors="ignore")
    # align with training: numeric-only (tanpa engineered features)
    df_row = df_row.select_dtypes(include=["number"]).astype("float32")

    # Reindex to training feature columns
    df_row = df_row.reindex(columns=feature_columns, fill_value=0.0)
    return df_row


sample_payload_idr = {
    "monthly_income": 5_000_000,
    "monthly_expense_total": 3_500_000,
    "budget_goal": 3_000_000,
    "credit_score": 720,
    "debt_to_income_ratio": 0.25,
    "loan_payment": 400_000,
    "investment_amount": 500_000,
    "subscription_services": 4,
    "emergency_fund": 2_500_000,
    "transaction_count": 60,
    "discretionary_spending": 800_000,
    "essential_spending": 2_200_000,
    "rent_or_mortgage": 1_200_000,
    "actual_savings": 1_500_000,
}

df_infer = build_features_from_raw_idr(sample_payload_idr)
X_inf = scaler.transform(df_infer.values)
probs = model.predict(X_inf, verbose=0)[0]
pred_id = int(np.argmax(probs))

out = {
    "classification": LABEL_MAPPING[str(pred_id)],
    "score": float(np.max(probs)),
    "probabilities": {LABEL_MAPPING[str(i)]: float(probs[i]) for i in range(3)},
}
out


{'classification': 'normal',
 'score': 0.5111116766929626,
 'probabilities': {'inflation': 0.18043749034404755,
  'normal': 0.5111116766929626,
  'recession': 0.3084507882595062}}

## 21. Production Notes

Integrasi produksi (FastAPI) sebaiknya:
- load `classification_model.keras`, `classification_scaler.joblib`, `classification_feature_columns.json`, `classification_label_mapping.json`
- susun feature vector sesuai `classification_feature_columns.json`
- `scaler.transform` → `model.predict` → mapping label English

Jika performa tidak mencapai `test_accuracy >= 0.85`, laporkan dengan jujur:
- kelas mana paling sulit
- confusion dominan (inflation↔recession, atau collapse ke normal)
- indikasi bahwa signal label vs fitur mungkin tidak cukup untuk 3-class classification
